In [ ]:
%pip install openai pyyaml

# Query Provisioned Throughput with Pay-Per-Token Fallback

Sends requests to a provisioned throughput endpoint first. On capacity errors (429, 503),
automatically retries against the equivalent pay-per-token model.

Configure `config.yml` before running:
- `databricks.host` — your workspace URL
- `model_serving.provisioned_endpoint` — name of your deployed PT endpoint
- `model_serving.pay_per_token_model` — the same underlying model available via pay-per-token

In [ ]:
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)

DATABRICKS_HOST      = config["databricks"]["host"]
SECRETS_SCOPE        = config["databricks"]["secrets_scope"]
TOKEN_SECRET_KEY     = config["databricks"]["token_secret_key"]
PROVISIONED_ENDPOINT = config["model_serving"]["provisioned_endpoint"]
PAY_PER_TOKEN_MODEL  = config["model_serving"]["pay_per_token_model"]
FALLBACK_STATUS_CODES = set(config["model_serving"]["fallback_status_codes"])

In [ ]:
import logging
from openai import OpenAI, RateLimitError, APIStatusError
from openai.types.chat import ChatCompletion, ChatCompletionChunk
from openai import Stream

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

client = OpenAI(
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
    api_key=dbutils.secrets.get(scope=SECRETS_SCOPE, key=TOKEN_SECRET_KEY),
)

In [ ]:
def query(
    messages: list[dict],
    stream: bool = False,
    **kwargs,
) -> ChatCompletion | Stream[ChatCompletionChunk]:
    try:
        response = client.chat.completions.create(
            model=PROVISIONED_ENDPOINT,
            messages=messages,
            stream=stream,
            **kwargs,
        )
        logger.info("route=provisioned endpoint=%s", PROVISIONED_ENDPOINT)
        return response

    except (RateLimitError, APIStatusError) as e:
        if getattr(e, "status_code", None) not in FALLBACK_STATUS_CODES:
            raise
        logger.warning(
            "route=pay_per_token model=%s reason=fallback status=%s",
            PAY_PER_TOKEN_MODEL,
            e.status_code,
        )
        return client.chat.completions.create(
            model=PAY_PER_TOKEN_MODEL,
            messages=messages,
            stream=stream,
            **kwargs,
        )

## Example Usage

In [ ]:
# Non-streaming
response = query(
    messages=[{"role": "user", "content": "What is the capital of France?"}]
)
print(response.choices[0].message.content)

In [ ]:
# Streaming
stream = query(
    messages=[{"role": "user", "content": "Explain provisioned throughput in one paragraph."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)